In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [3]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.')]

In [32]:
import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
llm=ChatGroq(model="Mixtral-8x7B-Instruct",groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7fb9c306cc40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fb9c30f3520>, model_name='Mixtral-8x7B-Instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
pip install langchain_huggingface 

Note: you may need to restart the kernel to use updated packages.


In [10]:
import langchain as lc

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [13]:
# vector. store
from langchain_chroma import Chroma


vector_store=Chroma.from_documents(documents,embedding=embeddings)

vector_store

In [14]:
vector_store.similarity_search("Which pet is the most independent?")

[Document(id='1b416196-09b8-497f-96ae-d55f790115d7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='c3aac1c6-ba59-4dc7-a116-a80b09cef5a7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='6afd200a-1038-42e6-8fc8-bc71931eb5d4', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='3f92eeb3-3d0d-4590-b86c-9364d0ccef5c', metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [15]:
#async query

await vector_store.asimilarity_search("cat")

[Document(id='1b416196-09b8-497f-96ae-d55f790115d7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='c3aac1c6-ba59-4dc7-a116-a80b09cef5a7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='6afd200a-1038-42e6-8fc8-bc71931eb5d4', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='13b9bad9-34d3-4603-b1e7-aaf60f4aa8ce', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

## Retrievers

In [16]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

In [21]:
retriever=RunnableLambda(vector_store.similarity_search).bind(k=2) #bind is for the top result 

In [22]:
retriever.batch(["cat","dog"])

[[Document(id='1b416196-09b8-497f-96ae-d55f790115d7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  Document(id='c3aac1c6-ba59-4dc7-a116-a80b09cef5a7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')],
 [Document(id='c3aac1c6-ba59-4dc7-a116-a80b09cef5a7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  Document(id='1b416196-09b8-497f-96ae-d55f790115d7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]]

In [23]:
retriever.invoke("rabbit")

[Document(id='6afd200a-1038-42e6-8fc8-bc71931eb5d4', metadata={'source': 'mammal-pets-doc'}, page_content='Rabbits are social animals that need plenty of space to hop around.'),
 Document(id='1b416196-09b8-497f-96ae-d55f790115d7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]

In [24]:
ret=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)


In [25]:
ret.batch(["cat","dog"])

[[Document(id='1b416196-09b8-497f-96ae-d55f790115d7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='c3aac1c6-ba59-4dc7-a116-a80b09cef5a7', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [33]:
# integrating with chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

res=rag_chain.invoke("tell me about dogs")
print(res.content)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `mixtral-8x7b-instruct` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}